#### Messages

Messages are the fundamental unit of context for models in LangChain. They represent the input and output of models, carrying both the content and metadata needed to represent the state of a conversation when interacting with an LLM.
Messages are objects that contain:
 - Role - Identifies the message type (e.g. system, user)
 - Content - Represents the actual content of the message (like text, images, audio, documents, etc.)
 - Metadata - Optional fields such as response information, message IDs, and token usage

LangChain provides a standard message type that works across all model providers, ensuring consistent behavior regardless of the model being called.

In LangChain, message types (roles) are represented by different message classes. The core ones are:

The main 5 message types

- HumanMessage (role: user / human): input from the user.
- AIMessage (role: assistant / ai): the model's response; can include tool_calls.
- SystemMessage (role: system): instructions that set the model's behavior or context.
- ToolMessage (role: tool): the result of a tool/function call, linked back via tool_call_id.
- FunctionMessage (role: function): older, deprecated predecessor of ToolMessage (OpenAI's legacy function calling).

Additional types

- ChatMessage: a generic message where you set any custom role string yourself.
- RemoveMessage: a special message used in LangGraph to delete a message from the state by ID (not sent to the model).

Chunk variants

For streaming, each has a chunk version: `HumanMessageChunk, AIMessageChunk, SystemMessageChunk, ToolMessageChunk, ChatMessageChunk`, and so on.

In [1]:
import os

from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:openai/gpt-oss-20b")

In [2]:
model.invoke("Please tell me how to properly cook basmati rice")

AIMessage(content='Below is a “cook‑book‑ready” method for stovetop basmati rice that works for most kitchens.  \nFeel free to tweak any step to match your taste or equipment.\n\n---\n\n## 1. Gather Your Tools\n\n| Item | Why it matters |\n|------|----------------|\n| **Medium‑to‑large pot** (or a rice cooker) with a tight‑fitting lid | Even heat distribution and good steam seal |\n| **Fine‑mesh sieve** | For rinsing and soaking |\n| **Measuring cups** | Precise water‑to‑rice ratio |\n| **Fork** | To fluff the grains after cooking |\n\n---\n\n## 2. Rinse (and Optional Soak)\n\n| Step | Details | Why |\n|------|---------|-----|\n| **Rinse** | Place rice in a fine‑mesh sieve and rinse under cold running water until the water runs clear. | Removes surface starch that can make rice gummy. |\n| **Soak (optional but recommended)** | Transfer rinsed rice to a bowl, cover with cold water, and let sit 20–30\u202fmin. Drain before cooking. | Soaking shortens cooking time and yields a fluffier te

### Text Prompts
Text prompts are strings - ideal for straightforward generation tasks where you don’t need to retain conversation history.

In [3]:
model.invoke("what is a roux")

AIMessage(content='A **roux** (pronounced “roo‑ee”) is a classic French cooking staple—a thickening agent made from equal parts flour and fat cooked together until it reaches a desired color and flavor. It’s the foundation of many sauces, gravies, soups, and stews.\n\n| **Component** | **Typical Ingredient** | **Purpose** |\n|---------------|------------------------|-------------|\n| **Fat** | Butter (most common), oil, or animal fat (e.g., bacon fat, lard) | Provides the medium that the flour absorbs; contributes flavor and texture. |\n| **Flour** | All‑purpose or whole‑meal flour | Supplies starch that gelatinizes when heated, thickening liquids. |\n\n### How It’s Made\n\n1. **Measure** equal parts by weight or volume (e.g., 1\u202fcup flour + 1\u202fcup butter).  \n2. **Melt** the fat in a saucepan over medium heat.  \n3. **Stir in** the flour, whisking continuously to avoid lumps.  \n4. **Cook** the mixture until it reaches the desired hue:\n   - **White roux** – cooked for 1–2\u20

Use text prompts when:
- You have a single, standalone request
- You don’t need conversation history
- You want minimal code complexity

### Message Prompts
Alternatively, you can pass in a list of messages to the model by providing a list of message objects.


Message types
- System message - Tells the model how to behave and provide context for interactions
- Human message - Represents user input and interactions with the model
- AI message - Responses generated by the model, including text content, tool calls, and metadata
- Tool message - Represents the outputs of tool calls

### System Message
A SystemMessage represent an initial set of instructions that primes the model’s behavior. You can use a system message to set the tone, define the model’s role, and establish guidelines for responses.


### Human Message
A HumanMessage represents user input and interactions. They can contain text, images, audio, files, and any other amount of multimodal content.

### AI Message
An AIMessage represents the output of a model invocation. They can include multimodal data, tool calls, and provider-specific metadata that you can later access.

### Tool Message
For models that support tool calling, AI messages can contain tool calls. Tool messages are used to pass the results of a single tool execution back to the model.

A `Runnable` is LangChain's standard interface: any component (models, prompts, parsers, retrievers, chains) that implements the same set of methods:

- invoke / ainvoke: single input
- stream / astream: streaming output
- batch / abatch: multiple inputs

In [14]:
from langchain.messages import AIMessage, HumanMessage, SystemMessage
import json

messages=[
    SystemMessage("You are a poetry expert"),
    HumanMessage("Write a poem about biryani")
]

# response=model.invoke(messages)
response=await model.ainvoke(messages)
response.content

'**Ode to Biryani**\n\nIn the hush of a kitchen, the pot begins to sing,  \nA whisper of cumin, a promise of spice,  \nThe clink of a spoon, the sizzle of a flame—  \nA fragrant orchestra, a culinary rite.  \n\nRed as the setting sun, the saffron threads unfurl,  \nThey mingle with cardamom, with cloves and with bay,  \nA golden river that glistens in the pot,  \nLike molten honey poured upon a bright day.  \n\nRice—soft, tender, each grain a tiny pearl—  \nIt swallows the aroma, it drinks the heat,  \nThe steam rises, a veil of incense and steam,  \nA hush of anticipation, a moment complete.  \n\nChicken or lamb, or a humble paneer,  \nThey tumble into the pot, a dance of flavors,  \nWith onions caramelized to a caramel hue,  \nAnd tomatoes that burst, their juice a ruby river.  \n\nThe pot is closed, a lid of hope and patience,  \nThe heat seeping through, a slow, patient kiss,  \nThe aroma spreads through the hallway, through the air—  \nA promise of nourishment, of comfort, of blis

In [16]:
print(json.dumps(response.model_dump(),indent=2))

{
  "content": "**Ode to Biryani**\n\nIn the hush of a kitchen, the pot begins to sing,  \nA whisper of cumin, a promise of spice,  \nThe clink of a spoon, the sizzle of a flame\u2014  \nA fragrant orchestra, a culinary rite.  \n\nRed as the setting sun, the saffron threads unfurl,  \nThey mingle with cardamom, with cloves and with bay,  \nA golden river that glistens in the pot,  \nLike molten honey poured upon a bright day.  \n\nRice\u2014soft, tender, each grain a tiny pearl\u2014  \nIt swallows the aroma, it drinks the heat,  \nThe steam rises, a veil of incense and steam,  \nA hush of anticipation, a moment complete.  \n\nChicken or lamb, or a humble paneer,  \nThey tumble into the pot, a dance of flavors,  \nWith onions caramelized to a caramel hue,  \nAnd tomatoes that burst, their juice a ruby river.  \n\nThe pot is closed, a lid of hope and patience,  \nThe heat seeping through, a slow, patient kiss,  \nThe aroma spreads through the hallway, through the air\u2014  \nA promise 

In [8]:
system_msg = SystemMessage("You are a helpful cooking assistant.")

messages = [
    system_msg,
    HumanMessage("How do I preheat a convection oven?")
]
response = model.invoke(messages)
print(response.content)

### How to Preheat a Convection Oven

Convection ovens work a bit differently from regular ovens because the fan circulates hot air, giving you faster, more even cooking. Preheating is still essential, but there are a few extra steps to keep in mind.

| Step | What to Do | Why It Matters |
|------|------------|----------------|
| **1. Set the temperature** | Turn the dial or use the digital control to the temperature your recipe calls for. If you’re following a recipe written for a conventional oven, lower the temperature by **25 °F (≈ 15 °C)**. | The fan makes the oven more efficient, so you need less heat to achieve the same result. |
| **2. Turn on the fan** | If your oven has a separate fan control, switch it to “convection” or “fan‑on.” Many modern ovens will automatically engage the fan when you set a temperature. | The circulating air cooks food evenly and reduces cooking time. |
| **3. Let it heat** | Allow the oven to run for **5–10 minutes** before putting food in. Most ovens

In [ ]:
## Detailed info to the LLM through System message
from langchain.messages import HumanMessage, SystemMessage

system_msg = SystemMessage("""
You are a senior chef with expertise in kitchen equipment and techniques.
Always provide clear steps and explain your reasoning.
Be concise but thorough in your explanations.
""")

messages = [
    system_msg,
    HumanMessage("How do I preheat a convection oven?")
]
response = model.invoke(messages)
print(response.content)

**Preheating a Convection Oven – Step‑by‑Step**

| Step | What to Do | Why It Matters |
|------|------------|----------------|
| **1. Set the temperature** | Turn the dial or keypad to the target temperature (e.g., 375 °F). | Convection ovens still need the same absolute temperature as a conventional oven for most recipes. |
| **2. Select “Convection” mode** | Press the convection button or toggle the fan on. | The fan circulates hot air, creating a more uniform temperature and faster heat transfer. |
| **3. Allow 10–15 minutes** | Leave the oven door closed and let it run until the indicator lights up or the digital display shows “Ready.” | Convection ovens often need a slightly longer preheat than conventional ovens because the fan helps distribute heat but also pulls air through the door. |
| **4. Verify the temperature** | Use an oven thermometer (or the built‑in probe) to confirm the interior matches the set value. | Even convection ovens can have minor temperature variations; a q

In [18]:
## Message Metadata
human_msg = HumanMessage(
    content="Hello!",
    name="ravi",  # Optional: identify different users
    id="msg_123",  # Optional: unique identifier for tracing
)

In [20]:
# Where msg_123 lives?
# msg_123 belongs to your HumanMessage. It stays on that object and is not copied into the response:

response = model.invoke([
  human_msg
])
# response
print(json.dumps(response.model_dump(),indent=2,ensure_ascii=False))

{
  "content": "Hello! How can I help you today?",
  "additional_kwargs": {
    "reasoning_content": "User says \"Hello!\" It's a greeting. We should respond politely."
  },
  "response_metadata": {
    "token_usage": {
      "completion_tokens": 33,
      "prompt_tokens": 73,
      "total_tokens": 106,
      "completion_time": 0.034190055,
      "completion_tokens_details": {
        "reasoning_tokens": 15
      },
      "prompt_time": 0.004156798,
      "prompt_tokens_details": null,
      "queue_time": 0.284153309,
      "total_time": 0.038346853
    },
    "model_name": "openai/gpt-oss-20b",
    "system_fingerprint": "fp_9340e7d14d",
    "service_tier": "on_demand",
    "finish_reason": "stop",
    "logprobs": null,
    "model_provider": "groq"
  },
  "type": "ai",
  "name": null,
  "id": "lc_run--01a22158-513f-708c-89c3-5dfc19bf6de6-0",
  "tool_calls": [],
  "invalid_tool_calls": [],
  "usage_metadata": {
    "input_tokens": 73,
    "output_tokens": 33,
    "total_tokens": 106,
  

#### Another example

In [21]:
from langchain.messages import HumanMessage, SystemMessage

# Create an AI message manually (e.g., for conversation history)
ai_msg = AIMessage("I'd be happy to help you with that question!, i remmember you were planning a trip to italy")

# Add to conversation history
messages = [
    SystemMessage("You are a helpful assistant"),
    HumanMessage("Can you help me?"),
    ai_msg,  # Insert as if it came from the model
    HumanMessage("Do you remmember for which local i was planning a trip?")
]

response = model.invoke(messages)
print(response.content)

Yes—I remember you mentioned you were planning a trip to **Italy**. If you’d like to discuss travel details, itineraries, or anything else related to your upcoming adventure, just let me know!


In [20]:
response.usage_metadata

{'input_tokens': 127,
 'output_tokens': 241,
 'total_tokens': 368,
 'output_token_details': {'reasoning': 197}}

#### Example ToolMessage

In [22]:
from langchain.messages import AIMessage, ToolMessage

# After a model makes a tool call
# (Here, we demonstrate manually creating the messages for brevity)
ai_message = AIMessage(
    content=[],
    tool_calls=[{
        "name": "check_ingredient_stock",
        "args": {"ingredient": "saffron"},
        "id": "call_123"
    }]
)

# Execute tool and create result message
stock_result = "In stock, 15g available"
tool_message = ToolMessage(
    content=stock_result,
    tool_call_id="call_123"  # Must match the call ID
)

# Continue conversation
messages = [
    HumanMessage("Is saffron in stock?"),
    ai_message,  # Model's tool call
    tool_message,  # Tool execution result
]
response = model.invoke(messages)  # Model processes the result

In [23]:
tool_message

ToolMessage(content='In stock, 15g available', tool_call_id='call_123')

In [24]:
response

AIMessage(content='Yes—saffron is in stock. We have **15\u202fg** available right now. If you need a specific amount or want to add it to your cart, just let me know!', additional_kwargs={'reasoning_content': 'We need to respond to the user: "Is saffron in stock?" The function returned that saffron is in stock, 15g available. The user just asked if saffron is in stock. We should respond accordingly, maybe ask how much they need. But likely we respond: Yes, saffron is in stock, 15g available. Offer to add to cart? The user didn\'t specify quantity. We can ask: "Would you like to add it to your cart?" Or simply confirm. I\'ll respond: Yes, saffron is in stock. 15g available. If you need a specific amount, let me know.'}, response_metadata={'token_usage': {'completion_tokens': 179, 'prompt_tokens': 116, 'total_tokens': 295, 'completion_time': 0.18938117, 'completion_tokens_details': {'reasoning_tokens': 130}, 'prompt_time': 0.016168408, 'prompt_tokens_details': None, 'queue_time': 0.37119

### 🤔 Before you move on
Every response so far has been free-form text (`response.content`) that you'd have to manually parse if you wanted structured data out of it. What if you want the model to hand back a *guaranteed* schema — a Pydantic object, a TypedDict, a dataclass — instead of a string you have to pick apart yourself? That's `5-structuredoutput.ipynb`.